# 🪄 SpellEngine — Dual-Task ASR + SER (Kaggle Edition)

A production-grade Kaggle notebook for a **speech-driven spell-casting game**: the player speaks an incantation, the engine
transcribes it (ASR) and reads the emotion in the voice (SER), and the two combine into a **Spell Power** score.

| Model | Architecture | Role |
|---|---|---|
| **A** | Conformer encoder (from scratch) + CTC head + attention-pooling SER head | Lightweight inference baseline (exported to ONNX) |
| **B** | Whisper-Small + LoRA (PEFT) + classification head | High-accuracy reference |

**Training tricks:** Kendall uncertainty weighting · OneCycleLR · AMP (mixed precision) · SpecAugment · Audiomentations (noise / gain / pitch) ·
strict checkpointing under `/kaggle/working/` with `resume_from_checkpoint`.

### 🛠 Setup before running
1. **Settings → Accelerator → GPU T4 x2** (or P100). CPU-only works but is slow.
2. **Settings → Internet → On** (needed for `pip` and to download the Whisper-Small weights).
3. **Add Input** (right sidebar → *+ Add Input*): search & add **`crema-d`** (CREMA-D emotional speech corpus, e.g. `ejlok1/cremad`).
4. Click **Save Version** at any point — `/kaggle/working/` artefacts are preserved between sessions.

### ⚡ Spell Power
$$\text{power} = 100 \times w_{\text{intent}}(\text{emotion}) \times p_{\text{emotion}} \times \big(0.4 + 0.6 \cdot \text{clarity}\big),\qquad \text{clarity} = 1 - \min(\text{WER}, 1)$$

where $w_{\text{intent}}$ is a per-emotion weight (anger 1.0 → neutral 0.4) and $p_{\text{emotion}}$ is the SER head's confidence.

Anger / fear / joy + clear diction ⇒ stronger spell. A mumbled incantation still keeps 40 % of its power.

## CELL 1 — Environment Setup

Kaggle's base image already has `torch`, `torchaudio`, `transformers`, `librosa` installed — we only `pip install` the deltas
(`peft`, `audiomentations`, `jiwer`, ONNX tooling) with `--no-deps`, and pin NumPy to 1.26.4 so the ONNX Runtime wheel keeps a matching ABI.

The cell checks what is actually installed in **this** container (not a flag file), so a fresh session re-installs automatically.
If it installs anything it stops and asks you to **Run → Restart Session**; re-run it afterwards and it continues.
Everything is written under `/kaggle/working/`, which is persisted with the notebook version.

In [ ]:
# ============================================================================
# CELL 1 — Environment Setup (Kaggle, NumPy-ABI-safe, restart-aware)
# ============================================================================
# ---- 1.1 Install whatever this container is missing ----
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.requirements import Requirement

NUMPY_PIN = "numpy==1.26.4"
REQUIRED = [                       # installed with --no-deps so nothing drags in another numpy
    "rapidfuzz", "accelerate", "safetensors", "huggingface-hub",
    "peft>=0.10.0",
    "audiomentations==0.35.0",
    "jiwer==3.0.4",
    "onnx==1.16.1",
    "onnxruntime==1.18.1",
    "onnxscript<0.2",
]

def _satisfied(spec: str) -> bool:
    req = Requirement(spec)
    try:
        return req.specifier.contains(version(req.name), prereleases=True)
    except PackageNotFoundError:
        return False

need_numpy = not _satisfied(NUMPY_PIN)
missing = [s for s in REQUIRED if not _satisfied(s)]

if need_numpy or missing:
    print("Installing dependencies (once per session, ~60s)…")
    if need_numpy:
        # Pin numpy first to avoid ABI drift with Kaggle's pre-built wheels
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        NUMPY_PIN, "--force-reinstall", "--no-deps"], check=True)
    if missing:
        print("  →", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *missing], check=True)
    print("\n" + "=" * 60)
    print("⚠ DEPENDENCIES INSTALLED — RESTART REQUIRED")
    print("=" * 60)
    print("Click: Run → Restart Session")
    print("Then re-run this cell — it will skip the install and continue.")
    print("=" * 60)
    raise SystemExit("Restart kernel now.")

print("✓ dependencies already installed, continuing")

# ---- 1.2 Standard imports ----
import json, time, math, random, gc, re, shutil, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch

# ---- 1.3 Project directory layout (Kaggle-native paths) ----
WORKING = "/kaggle/working"
INPUT = "/kaggle/input"
PROJECT_ROOT = f"{WORKING}/SpellEngine"
DIRS = {
    "root":       PROJECT_ROOT,
    "manifests":  f"{PROJECT_ROOT}/manifests",
    "ckpts":      f"{PROJECT_ROOT}/checkpoints",
    "ckpts_conf": f"{PROJECT_ROOT}/checkpoints/conformer",
    "ckpts_whis": f"{PROJECT_ROOT}/checkpoints/whisper_lora",
    "exports":    f"{PROJECT_ROOT}/exports",
    "logs":       f"{PROJECT_ROOT}/logs",
    "cache":      f"{PROJECT_ROOT}/hf_cache",
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

# Route the Hugging Face cache into /kaggle/working so model weights survive a re-run
os.environ["HF_HOME"] = DIRS["cache"]
os.environ["TRANSFORMERS_CACHE"] = DIRS["cache"]
os.environ["HF_DATASETS_CACHE"] = DIRS["cache"]

# ---- 1.4 Reproducibility ----
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- 1.5 Device + capability check ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"PyTorch         : {torch.__version__}")
print(f"NumPy           : {np.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}           : {p.name} ({p.total_memory/1e9:.1f} GB)")
print(f"Mixed precision : {USE_AMP}")
print(f"Project root    : {PROJECT_ROOT}")
print(f"Inputs visible  : {os.listdir(INPUT) if os.path.isdir(INPUT) else 'NONE'}")

### (Optional) GPU / PyTorch build diagnostics
Confirms the installed PyTorch wheel was compiled for your GPU's architecture (T4 = `sm_75`, P100 = `sm_60`).

In [ ]:
import torch, subprocess

print("=" * 60); print("PYTORCH BUILD"); print("=" * 60)
print(f"torch         : {torch.__version__}")
print(f"CUDA build    : {torch.version.cuda}")
print(f"compiled archs: {torch.cuda.get_arch_list()}")

print("\n" + "=" * 60); print("YOUR GPU(S)"); print("=" * 60)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  sm_{p.major}{p.minor}  ({p.total_memory/1e9:.1f} GB)")

print("\n" + "=" * 60); print("NVIDIA-SMI"); print("=" * 60)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                          "--format=csv"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not available (CPU session)")

## CELL 2 — Data Discovery & Manifest Building

Kaggle datasets live under `/kaggle/input/<owner>/<dataset-slug>/`. Nothing is downloaded here — the cell walks `/kaggle/input`
for an `AudioWAV/` folder of `XXXX_XXX_XXX_XX.wav` files, so it works with whichever CREMA-D mirror you added.

CREMA-D gives every clip **both** a transcript (one of 12 fixed sentences) **and** an emotion label, so each clip becomes one ASR row
and one SER row.

`SPLIT_BY = "file"` reproduces the original run (clip-level split). `"speaker"` keeps every actor in a single split, which is the
stricter, speaker-independent evaluation — if you switch, delete `checkpoints/` first so training starts fresh on the new split.

In [ ]:
# ============================================================================
# CELL 2 — Data Discovery & Manifest Building (CREMA-D, self-contained)
# ============================================================================
import os, re
import pandas as pd
from pathlib import Path

# Defensive: re-define what later cells need, in case the kernel was restarted
SEED = 1337
WORKING = "/kaggle/working"
PROJECT_ROOT = f"{WORKING}/SpellEngine"
DIRS = {
    "root":       PROJECT_ROOT,
    "manifests":  f"{PROJECT_ROOT}/manifests",
    "ckpts":      f"{PROJECT_ROOT}/checkpoints",
    "ckpts_conf": f"{PROJECT_ROOT}/checkpoints/conformer",
    "ckpts_whis": f"{PROJECT_ROOT}/checkpoints/whisper_lora",
    "exports":    f"{PROJECT_ROOT}/exports",
    "logs":       f"{PROJECT_ROOT}/logs",
    "cache":      f"{PROJECT_ROOT}/hf_cache",
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

SPLIT_BY = "file"        # "file" (original run) | "speaker" (speaker-independent)
INPUT_DIR = Path("/kaggle/input")

# ----- Find CREMA-D -----
def find_crema_root():
    for d in INPUT_DIR.rglob("AudioWAV"):
        if d.is_dir() and any(d.glob("*.wav")):
            return d
    return None

crema_root = find_crema_root()
print(f"CREMA-D root: {crema_root}")
assert crema_root is not None, "CREMA-D not found! Add it via the Input panel (+ Add Input → 'crema-d')."

# ----- Build the CREMA-D manifest with BOTH tasks -----
EMO = {"NEU": "neutral", "HAP": "happy", "SAD": "sad", "ANG": "anger", "FEA": "fear", "DIS": "disgust"}
SENT = {
    "IEO": "it's eleven o'clock",            "TIE": "that is exactly what happened",
    "IOM": "i'm on my way to the meeting",   "IWW": "i wonder what this is about",
    "TAI": "the airplane is almost full",    "MTI": "maybe tomorrow it will be cold",
    "IWL": "i would like a new alarm clock", "ITH": "i think i have a doctor's appointment",
    "DFA": "don't forget a jacket",          "ITS": "i think i've seen this before",
    "TSI": "the surface is slick",           "WSI": "we'll stop in a couple of minutes",
}

rows = []
for wav in crema_root.glob("*.wav"):
    parts = wav.stem.split("_")
    if len(parts) != 4:
        continue
    actor, sent_code, emo_code, _ = parts
    text = SENT.get(sent_code, "")
    emotion = EMO.get(emo_code, "neutral")
    if not text:
        continue
    # CREMA-D has known transcripts AND emotion labels —
    # so each clip serves as BOTH an ASR and an SER training example.
    for task in ("asr", "ser"):
        rows.append({"path": str(wav), "text": text, "emotion": emotion,
                     "task": task, "speaker": actor})

crema_df = pd.DataFrame(rows)
print(f"Built {len(crema_df):,} (clip, task) rows from CREMA-D")

# Deterministic 80/10/10 split by FILE (default) or by SPEAKER, so a unit never spans splits
split_key = {"file": "path", "speaker": "speaker"}[SPLIT_BY]
units = pd.Series(crema_df[split_key].unique()).sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(units)
train_units = set(units[:int(0.8 * n)])
dev_units   = set(units[int(0.8 * n):int(0.9 * n)])

def assign_split(u):
    if u in train_units: return "train"
    if u in dev_units:   return "dev"
    return "test"

crema_df["split"] = crema_df[split_key].map(assign_split)
manifest = crema_df.copy()

manifest_path = Path(DIRS["manifests"]) / "spell_engine_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"\nSplit by        : {SPLIT_BY}")
print(f"Total rows      : {len(manifest):,}")
print(manifest.groupby(["task", "split"]).size().unstack(fill_value=0))
print("\nSpeakers / split:", manifest.groupby("split")["speaker"].nunique().to_dict())
print(f"\nManifest saved → {manifest_path}")
assert len(manifest) > 0, "Empty manifest!"

## CELL 3 — Data Pipeline (Lazy Loading, SpecAugment, Audiomentations)

A truly lazy `Dataset`: each `__getitem__` reads exactly one audio file from `/kaggle/input/`, applies augmentation, and returns
features + labels. Nothing is held in RAM, so corpora far larger than Kaggle's 16 GB host RAM stay tractable.

Waveform-level augmentations (noise, gain, pitch) simulate a player's bedroom acoustics. SpecAugment then masks bands of the
Mel spectrogram for ASR robustness. `load_audio`, `conformer_features` and `normalize_text` are shared with the evaluation,
export and inference cells so training and play-time preprocessing can never drift apart.

In [ ]:
# ============================================================================
# CELL 3 — Data Pipeline (Lazy Loading, SpecAugment, Audiomentations)
# ============================================================================
import re, string, warnings, logging
import numpy as np, pandas as pd, torch, torchaudio
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from audiomentations import Compose, AddGaussianNoise, PitchShift, Gain

# Silence any leftover audiomentations chatter
warnings.filterwarnings("ignore")
logging.getLogger("audiomentations").setLevel(logging.ERROR)

SR = 16000
N_MELS = 80
N_FFT = 400
HOP = 160
MIN_SAMPLES = SR // 10           # pad anything shorter than 100 ms (guards empty / truncated files)

EMOTIONS = ["neutral", "happy", "sad", "anger", "fear", "disgust"]
EMO2IDX = {e: i for i, e in enumerate(EMOTIONS)}
IDX2EMO = {i: e for e, i in EMO2IDX.items()}

# ----- 3.1 Char tokenizer for the Conformer CTC head -----
class CharTokenizer:
    def __init__(self):
        self.blank = 0
        chars = list(" '") + list(string.ascii_lowercase)
        self.itos = ["<blank>"] + chars
        self.stoi = {c: i for i, c in enumerate(self.itos)}

    @property
    def vocab_size(self): return len(self.itos)

    def encode(self, text: str):
        text = text.lower()
        return [self.stoi[c] for c in text if c in self.stoi]

    def decode(self, ids, collapse=True):
        out, prev = [], None
        for i in ids:
            if collapse and i == prev: continue
            prev = i
            if i == self.blank: continue
            out.append(self.itos[i])
        return "".join(out)

tokenizer = CharTokenizer()
print(f"Char vocab size: {tokenizer.vocab_size}")

# ----- 3.2 Text normalisation (applied to references AND hypotheses before WER) -----
_VALID_CHARS = set(" '" + string.ascii_lowercase)

def normalize_text(s) -> str:
    """lowercase, curly→straight apostrophes, drop punctuation, collapse whitespace."""
    s = str(s).lower().replace("’", "'").replace("-", " ")
    s = "".join(c for c in s if c in _VALID_CHARS)
    return re.sub(r"\s+", " ", s).strip()

# ----- 3.3 Audio I/O + Conformer features (shared by train / eval / export / play) -----
def load_audio(path) -> torch.Tensor:
    """Mono float32 waveform at 16 kHz, shape (samples,)."""
    try:
        wav, sr = torchaudio.load(str(path))                     # (channels, samples)
    except Exception:
        import soundfile as sf
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)   # (samples, channels)
        wav = torch.from_numpy(data.T.copy())
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = wav.squeeze(0).float()
    if wav.numel() < MIN_SAMPLES:
        wav = F.pad(wav, (0, MIN_SAMPLES - wav.numel()))
    return wav

_mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0)
_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

def conformer_features(wav: torch.Tensor) -> torch.Tensor:
    """(samples,) → per-utterance normalised log-mel, shape (n_mels, frames)."""
    mel = _to_db(_mel(wav) + 1e-9)
    return (mel - mel.mean()) / (mel.std() + 1e-5)

# ----- 3.4 Augmentation (no RoomSimulator — it needs pyroomacoustics) -----
wave_aug = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.4),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
])

def spec_augment(mel: torch.Tensor, F=27, T=40, num_F=2, num_T=2):
    """SpecAugment on (n_mels, frames); returns a masked copy."""
    mel = mel.clone()
    n_mels, n_frames = mel.shape
    for _ in range(num_F):
        f = np.random.randint(0, F + 1); f0 = np.random.randint(0, max(1, n_mels - f))
        mel[f0:f0 + f, :] = 0
    for _ in range(num_T):
        t = np.random.randint(0, T + 1); t0 = np.random.randint(0, max(1, n_frames - t))
        mel[:, t0:t0 + t] = 0
    return mel

# ----- 3.5 The Dataset -----
class SpellEngineDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, split: str,
                 augment: bool = False, max_seconds: float = 16.0,
                 use_whisper_features: bool = False, whisper_extractor=None):
        self.df = manifest_df[manifest_df["split"] == split].reset_index(drop=True)
        self.augment = augment
        self.max_samples = int(max_seconds * SR)
        self.use_whisper_features = use_whisper_features
        self.whisper_extractor = whisper_extractor

    def __len__(self): return len(self.df)

    def _load_audio(self, path):
        wav = load_audio(path)
        if wav.numel() > self.max_samples:
            start = np.random.randint(0, wav.numel() - self.max_samples) if self.augment else 0
            wav = wav[start:start + self.max_samples]
        return wav

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = self._load_audio(row["path"])

        if self.augment:
            try:
                wav = torch.from_numpy(wave_aug(samples=wav.numpy(), sample_rate=SR)).float()
            except Exception:
                pass

        if self.use_whisper_features:
            feats = self.whisper_extractor(
                wav.numpy(), sampling_rate=SR, return_tensors="pt"
            ).input_features.squeeze(0)
        else:
            feats = conformer_features(wav)
            if self.augment:
                feats = spec_augment(feats)

        text_ids = torch.tensor(tokenizer.encode(str(row["text"])), dtype=torch.long)
        emo_id = torch.tensor(EMO2IDX[row["emotion"]], dtype=torch.long)
        return {
            "features": feats, "feat_len": feats.shape[-1],
            "text_ids": text_ids, "text_len": len(text_ids),
            "emotion": emo_id, "task": row["task"], "path": row["path"],
        }

# ----- 3.6 Collate (pads variable-length) -----
def collate_conformer(batch):
    max_T = max(b["feat_len"] for b in batch)
    max_L = max(max(b["text_len"] for b in batch), 1)
    n_mels = batch[0]["features"].shape[0]
    feats = torch.zeros(len(batch), n_mels, max_T)
    feat_lens = torch.zeros(len(batch), dtype=torch.long)
    texts = torch.zeros(len(batch), max_L, dtype=torch.long)
    text_lens = torch.zeros(len(batch), dtype=torch.long)
    emotions = torch.zeros(len(batch), dtype=torch.long)
    tasks, paths = [], []
    for i, b in enumerate(batch):
        T = b["feat_len"]
        feats[i, :, :T] = b["features"]
        feat_lens[i] = T
        L = b["text_len"]
        if L > 0: texts[i, :L] = b["text_ids"]
        text_lens[i] = L
        emotions[i] = b["emotion"]
        tasks.append(b["task"]); paths.append(b["path"])
    return {"features": feats, "feat_lens": feat_lens, "texts": texts,
            "text_lens": text_lens, "emotions": emotions, "tasks": tasks, "paths": paths}

def collate_whisper(batch):
    feats = torch.stack([b["features"] for b in batch])        # Whisper pads every clip to 30 s
    max_L = max(max(b["text_len"] for b in batch), 1)
    texts = torch.zeros(len(batch), max_L, dtype=torch.long)
    text_lens = torch.zeros(len(batch), dtype=torch.long)
    emotions = torch.zeros(len(batch), dtype=torch.long)
    tasks, paths = [], []
    for i, b in enumerate(batch):
        L = b["text_len"]
        if L > 0: texts[i, :L] = b["text_ids"]
        text_lens[i] = L
        emotions[i] = b["emotion"]
        tasks.append(b["task"]); paths.append(b["path"])
    return {"features": feats, "feat_lens": None, "texts": texts,
            "text_lens": text_lens, "emotions": emotions, "tasks": tasks, "paths": paths}

manifest = pd.read_csv(Path(DIRS["manifests"]) / "spell_engine_manifest.csv")
print(f"Manifest rows: {len(manifest):,}")
print(manifest.groupby(["task", "split"]).size().unstack(fill_value=0))

# Smoke-test one item
_ds = SpellEngineDataset(manifest, "train", augment=False)
_item = _ds[0]
print(f"\n✓ first item: feats={tuple(_item['features'].shape)}, text_len={_item['text_len']}, task={_item['task']}")
del _ds, _item

## CELL 4 — Visual EDA

Sanity check before training: waveform & log-Mel for one ASR and one SER sample, emotion class balance, transcript-length histogram.
Catch corpus pathologies (extreme imbalance, silent files) now rather than after a 6-hour run.

In [ ]:
# ============================================================================
# CELL 4 — Visual EDA
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, librosa.display

sns.set_theme(style="whitegrid")

# 4.1 Waveform vs Mel-spectrogram, one (different) sample per task
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for col, task in enumerate(["asr", "ser"]):
    sub = manifest[manifest["task"] == task]
    if len(sub) == 0:
        for r in range(2): axes[r, col].set_visible(False)
        continue
    sample = sub.sample(1, random_state=SEED + col).iloc[0]
    wav = load_audio(sample["path"]).numpy()

    axes[0, col].plot(np.arange(len(wav)) / SR, wav, lw=0.5)
    axes[0, col].set_title(f"{task.upper()} — waveform\n[{sample['emotion']}] \"{str(sample['text'])[:60]}\"")
    axes[0, col].set_xlabel("time (s)"); axes[0, col].set_ylabel("amp")

    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[1, col])
    axes[1, col].set_title(f"{task.upper()} — log-mel spectrogram")
    fig.colorbar(img, ax=axes[1, col], format="%+2.f dB")
plt.tight_layout(); plt.show()

# 4.2 Emotion distribution + transcript-length histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ser_sub = manifest[manifest["task"] == "ser"]
if len(ser_sub) > 0:
    emo_counts = ser_sub["emotion"].value_counts()
    sns.barplot(x=emo_counts.index, y=emo_counts.values, hue=emo_counts.index,
                palette="viridis", legend=False, ax=axes[0])
    axes[0].set_title("Emotion distribution (SER subset)"); axes[0].set_ylabel("clips")
    for i, v in enumerate(emo_counts.values):
        axes[0].text(i, v + max(emo_counts.values) * 0.01, str(v), ha="center")
else:
    axes[0].set_visible(False)

text_lens = manifest[manifest["task"] == "asr"]["text"].fillna("").astype(str).str.len()
if len(text_lens) > 0:
    axes[1].hist(text_lens, bins=50, color="#4c72b0", edgecolor="white")
    axes[1].set_title("Transcript char-length distribution (ASR)")
    axes[1].set_xlabel("characters"); axes[1].set_ylabel("count")
    axes[1].axvline(text_lens.median(), color="red", ls="--", label=f"median = {text_lens.median():.0f}")
    axes[1].legend()
else:
    axes[1].set_visible(False)
plt.tight_layout(); plt.show()

print(f"ASR clips : {(manifest['task'] == 'asr').sum():,}")
print(f"SER clips : {(manifest['task'] == 'ser').sum():,}")
if len(text_lens) > 0:
    print(f"Median transcript chars: {text_lens.median():.0f} | max: {text_lens.max()}")

## CELL 5 — Model A: Conformer from Scratch

Clean PyTorch Conformer encoder.

**Per-block flow:** ½ feed-forward → multi-head self-attention → depthwise-separable convolution → ½ feed-forward → LayerNorm.
Every sub-module wraps a residual connection — what makes Conformer train without the warm-up pain of vanilla Transformers on speech.

**Dim check:** `d_model=144`, `n_heads=4` ⇒ `d_head = 144/4 = 36` ✓. The conv module's `expansion=2` projects 144 → 288 → GLU → 144,
preserving the residual shape. Output lengths are computed with tensor ops (no Python loop), which also keeps the model ONNX-exportable.

In [ ]:
# ============================================================================
# CELL 5 — Model A: Conformer from Scratch
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F

class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class FeedForwardModule(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * expansion), Swish(), nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, f"d_model={d_model} not divisible by heads={num_heads}"
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x_n = self.norm(x)
        out, _ = self.attn(x_n, x_n, x_n, key_padding_mask=mask, need_weights=False)
        return self.drop(out)

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, expansion=2, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        inner = d_model * expansion
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Conv1d(d_model, inner * 2, 1)
        self.dw = nn.Conv1d(inner, inner, kernel_size, padding=kernel_size // 2, groups=inner)
        self.bn = nn.BatchNorm1d(inner)
        self.act = Swish()
        self.pw2 = nn.Conv1d(inner, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x).transpose(1, 2)
        x = F.glu(self.pw1(x), dim=1)
        x = self.act(self.bn(self.dw(x)))
        x = self.pw2(x)
        return self.drop(x.transpose(1, 2))

class ConformerBlock(nn.Module):
    def __init__(self, d_model=144, num_heads=4, ff_expansion=4, conv_kernel=31, dropout=0.1):
        super().__init__()
        self.ff1 = FeedForwardModule(d_model, ff_expansion, dropout)
        self.mhsa = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.conv = ConformerConvModule(d_model, conv_kernel, dropout=dropout)
        self.ff2 = FeedForwardModule(d_model, ff_expansion, dropout)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        x = x + 0.5 * self.ff1(x)
        x = x + self.mhsa(x, mask=mask)
        x = x + self.conv(x)
        x = x + 0.5 * self.ff2(x)
        return self.norm(x)

class ConvSubsampler(nn.Module):
    """4× temporal subsampling — turns ~10 ms frames into ~40 ms."""
    def __init__(self, in_channels, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, d_model, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(d_model, d_model, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.proj = nn.Linear(d_model * (((in_channels + 1) // 2 + 1) // 2), d_model)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.net(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
        return self.proj(x)

class AttentionPool(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.score = nn.Linear(d_model, 1)
    def forward(self, x, mask=None):
        s = self.score(x).squeeze(-1)
        if mask is not None: s = s.masked_fill(mask, -1e4)
        w = F.softmax(s, dim=-1).unsqueeze(-1)
        return (x * w).sum(dim=1)

class ConformerSpellEngine(nn.Module):
    def __init__(self, n_mels=80, d_model=144, num_layers=8, num_heads=4,
                 conv_kernel=31, dropout=0.1, vocab_size=29, num_emotions=6):
        super().__init__()
        self.subsample = ConvSubsampler(n_mels, d_model)
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, num_heads, 4, conv_kernel, dropout)
            for _ in range(num_layers)])
        self.ctc_head = nn.Linear(d_model, vocab_size)
        self.emo_pool = AttentionPool(d_model)
        self.emo_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, num_emotions))

    @staticmethod
    def _len_after_subsample(L):
        """Two stride-2 convs (k=3, p=1). Works on Python ints and on LongTensors."""
        return ((L + 1) // 2 + 1) // 2

    def _make_mask(self, lens, max_T):
        return torch.arange(max_T, device=lens.device)[None, :] >= lens[:, None]

    def forward(self, feats, feat_lens):
        x = self.subsample(feats)
        x = self.dropout(x)
        new_lens = self._len_after_subsample(feat_lens.to(x.device).long())
        mask = self._make_mask(new_lens, x.size(1))
        for blk in self.layers: x = blk(x, mask=mask)
        ctc_logits = self.ctc_head(x)
        pooled = self.emo_pool(x, mask=mask)
        emo_logits = self.emo_head(pooled)
        return ctc_logits, emo_logits, new_lens

# Sanity check
_m = ConformerSpellEngine(vocab_size=tokenizer.vocab_size, num_emotions=len(EMOTIONS))
_x = torch.randn(2, 80, 400); _l = torch.tensor([400, 350])
_c, _e, _nl = _m(_x, _l)
print(f"CTC logits   : {_c.shape}")
print(f"Emo logits   : {_e.shape}")
print(f"Sub-sampled L: {_nl.tolist()}")
print(f"Params       : {sum(p.numel() for p in _m.parameters())/1e6:.2f} M")
del _m, _x, _l, _c, _e, _nl

## CELL 6 — Model B: Whisper-Small + LoRA

Whisper-Small (~244 M params) wrapped with LoRA so only ~0.7 % of the weights are updated. The SER head reads mean-pooled encoder
states; the LM head & decoder stay intact for ASR. The encoder runs **once** per step and its output feeds both the decoder and
the SER head (the encoder is most of Whisper's compute, so this roughly halves training time).

> **Note:** Internet must be **On** for the first run (downloads Whisper weights into `/kaggle/working/SpellEngine/hf_cache/`).

In [ ]:
# ============================================================================
# CELL 6 — Model B: Whisper-Small + LoRA
# ============================================================================
import gc
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
from transformers.utils import logging as hf_logging
from peft import LoraConfig, get_peft_model, TaskType

# generate() otherwise prints the max_length / logits-processor warnings once per batch
hf_logging.set_verbosity_error()

WHISPER_NAME = "openai/whisper-small"
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_NAME, cache_dir=DIRS["cache"])
whisper_processor.tokenizer.set_prefix_tokens(language="english", task="transcribe")

class WhisperSpellEngine(nn.Module):
    def __init__(self, model_name=WHISPER_NAME, num_emotions=6,
                 lora_r=16, lora_alpha=32, lora_dropout=0.05):
        super().__init__()
        self.whisper = WhisperForConditionalGeneration.from_pretrained(
            model_name, cache_dir=DIRS["cache"])
        self.whisper.generation_config.language = "en"
        self.whisper.generation_config.task = "transcribe"
        self.whisper.generation_config.forced_decoder_ids = None

        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
            target_modules=["q_proj", "v_proj"],
        )
        self.whisper = get_peft_model(self.whisper, lora_cfg)
        self.whisper.print_trainable_parameters()

        d = self.whisper.base_model.model.config.d_model
        self.emo_head = nn.Sequential(
            nn.LayerNorm(d), nn.Linear(d, d // 2), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(d // 2, num_emotions),
        )

    @property
    def core(self):
        """The underlying WhisperForConditionalGeneration (LoRA layers are injected in place)."""
        return self.whisper.base_model.model

    def encode(self, input_features):
        return self.core.get_encoder()(input_features).last_hidden_state

    def forward(self, input_features, labels=None):
        enc = self.encode(input_features)                       # encoder runs ONCE …
        asr_out = self.core(encoder_outputs=BaseModelOutput(last_hidden_state=enc),
                            labels=labels)                      # … feeds the decoder …
        emo_logits = self.emo_head(enc.mean(dim=1))             # … and the SER head
        return asr_out, emo_logits

    @torch.no_grad()
    def transcribe(self, input_features, max_new_tokens=128):
        gen = self.whisper.generate(input_features=input_features, max_new_tokens=max_new_tokens)
        return whisper_processor.batch_decode(gen, skip_special_tokens=True)

print("Loading Whisper-Small + applying LoRA …")
_w = WhisperSpellEngine(num_emotions=len(EMOTIONS))
_dummy = torch.randn(1, 80, 3000)
_asr_out, _emo = _w(_dummy, labels=torch.randint(0, 51865, (1, 8)))
print(f"ASR loss   : {_asr_out.loss.item():.3f}")
print(f"Emo logits : {_emo.shape}")
del _w, _dummy, _asr_out, _emo; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## CELL 7 — Multi-Task Loss (Kendall) + Checkpoint Manager

**Kendall uncertainty weighting** (Kendall, Gal & Cipolla, 2018): each task's loss weight is a learnable $s_i = \log\sigma_i^2$.
The loss actually optimised is

$$\mathcal{L} \;=\; \sum_{i\,\in\,\{\text{asr},\,\text{ser}\}} \frac{1}{\sigma_i^2}\,\mathcal{L}_i \;+\; \log\sigma_i
\;=\; \sum_i e^{-s_i}\,\mathcal{L}_i + \tfrac{1}{2}s_i$$

The log terms regularise σ so the network can't trivially zero out the weights — this removes the task-balance hyper-parameter entirely.

**Checkpoint manager** writes model + optimizer + scaler + scheduler + epoch + mtl_state to `/kaggle/working/` after every epoch
(atomically, so a session that dies mid-save never leaves a corrupt `latest.pt`). `resume_from_checkpoint` picks up exactly where
you stopped — critical because Kaggle's 12 h session limit means long runs span several sessions.

In [ ]:
# ============================================================================
# CELL 7 — Multi-Task Loss (Kendall) + Checkpoint Manager
# ============================================================================
import os, shutil
import numpy as np
from pathlib import Path

class MultiTaskUncertaintyLoss(nn.Module):
    """Kendall et al. (2018). The log-variance parameterisation avoids exp underflow."""
    def __init__(self, num_tasks=2):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, losses):
        total = 0.0
        for i, L in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total = total + precision * L + 0.5 * self.log_vars[i]
        return total

    def weights(self):
        with torch.no_grad():
            return torch.exp(-self.log_vars).cpu().tolist()

def _plain(v):
    """numpy scalars → Python numbers so checkpoints stay portable."""
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.integer):  return int(v)
    return v

class CheckpointManager:
    """Writes EVERY epoch + a 'latest' pointer + a 'best' pointer (keeps the last `max_keep` epochs)."""
    def __init__(self, ckpt_dir: str, max_keep: int = 3):
        self.dir = Path(ckpt_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.latest = self.dir / "latest.pt"
        self.best = self.dir / "best.pt"

    @staticmethod
    def _atomic_save(obj, path: Path):
        tmp = path.with_name(path.name + ".tmp")
        torch.save(obj, tmp); os.replace(tmp, path)

    @staticmethod
    def _atomic_copy(src: Path, dst: Path):
        tmp = dst.with_name(dst.name + ".tmp")
        shutil.copyfile(src, tmp); os.replace(tmp, dst)

    def save(self, *, model, optimizer, scheduler, scaler, mtl,
             epoch: int, metrics: dict, is_best: bool = False):
        state = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler else None,
            "scaler_state": scaler.state_dict() if scaler else None,
            "mtl_state": mtl.state_dict() if mtl else None,
            "metrics": {k: _plain(v) for k, v in metrics.items()},
        }
        path = self.dir / f"epoch_{epoch:03d}.pt"
        self._atomic_save(state, path)
        self._atomic_copy(path, self.latest)
        if is_best: self._atomic_copy(path, self.best)
        print(f"💾 saved {path.name} (best={is_best})")
        self._rotate()

    def _rotate(self):
        epoch_files = sorted(self.dir.glob("epoch_*.pt"))
        for old in epoch_files[:-self.max_keep]:
            old.unlink(missing_ok=True)

    def resume(self, model, optimizer=None, scheduler=None, scaler=None, mtl=None,
               path=None, map_location="cpu"):
        target = Path(path) if path else self.latest
        if not target.exists():
            print("ℹ no checkpoint found — starting fresh"); return 0, {}
        state = torch.load(target, map_location=map_location, weights_only=False)  # our own files
        model.load_state_dict(state["model_state"], strict=False)
        if optimizer and state.get("optimizer_state"): optimizer.load_state_dict(state["optimizer_state"])
        if scheduler and state.get("scheduler_state"): scheduler.load_state_dict(state["scheduler_state"])
        if scaler and state.get("scaler_state"):       scaler.load_state_dict(state["scaler_state"])
        if mtl and state.get("mtl_state"):             mtl.load_state_dict(state["mtl_state"])
        epoch = state["epoch"] + 1
        print(f"⤴ resumed from {target.name} → starting epoch {epoch}")
        return epoch, state.get("metrics", {})

def resume_from_checkpoint(model, ckpt_dir, optimizer=None, scheduler=None,
                           scaler=None, mtl=None, path=None):
    return CheckpointManager(ckpt_dir).resume(
        model, optimizer, scheduler, scaler, mtl,
        path=path, map_location=DEVICE)

def load_weights(model, path, map_location=None):
    """Load only the model weights of a checkpoint (for evaluation / export)."""
    state = torch.load(path, map_location=map_location or DEVICE, weights_only=False)
    missing, _ = model.load_state_dict(state["model_state"], strict=False)
    m = state.get("metrics", {})
    print(f"✓ loaded {Path(path).name}: epoch {state['epoch'] + 1}, "
          f"val WER={m.get('wer', float('nan')):.3f}, F1={m.get('f1', float('nan')):.3f}"
          + (f"  ({len(missing)} missing keys)" if missing else ""))
    return m

print("✓ MultiTaskUncertaintyLoss & CheckpointManager defined")

## CELL 8 — Master Training Loop (AMP, OneCycleLR, Live Spell-Power Demo)

A single trainer drives either model. Per epoch it logs:
- training loss broken down by task,
- validation **WER, CER** (`jiwer`, on normalised text), **macro-F1 & accuracy** for emotions (`scikit-learn`),
- live **Kendall weights** so you can watch the network self-balance the tasks,
- 🔮 a **Spell Power** prediction on a random validation clip — the whole point of the project.

**Tips**
- Drop `EPOCHS_CONF` / `EPOCHS_WHIS` to `1` for a smoke test before committing to a long run.
- Training resumes from `latest.pt`. The OneCycle schedule length is restored from the checkpoint, so if you **change the number of
  epochs** after a run has started, pass `resume=False` (or delete that model's `checkpoints/` folder) to start a fresh schedule.
- Whisper labels drop the leading `<|startoftranscript|>`: the model prepends it itself when building decoder inputs, so keeping it
  would teach a doubled start token. A Whisper checkpoint trained before this fix only benefits after retraining.

In [ ]:
# ============================================================================
# CELL 8a — Spell power, evaluation and the trainer
# ============================================================================
import random
import pandas as pd
from jiwer import wer as jiwer_wer, cer as jiwer_cer
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# ---------- 8.1 spell power formula ----------
INTENT_WEIGHTS = {"anger": 1.0, "fear": 0.9, "happy": 0.85,
                  "disgust": 0.7, "sad": 0.5, "neutral": 0.4}

def spell_power(emotion_label, emotion_conf, wer_value):
    intent = INTENT_WEIGHTS.get(emotion_label, 0.5)
    clarity = max(0.0, 1.0 - min(wer_value, 1.0))
    return round(100 * intent * emotion_conf * (0.4 + 0.6 * clarity), 1)

def clip_wer(ref, hyp):
    if not ref: return 1.0
    if not hyp: return 1.0            # everything deleted
    return float(jiwer_wer(ref, hyp))

# ---------- 8.2 evaluation ----------
def _new_dump():
    return {"refs": [], "hyps": [], "asr_paths": [],
            "y_true": [], "y_pred": [], "y_conf": [], "ser_paths": []}

def summarize(dump):
    metrics = {}
    if dump["refs"]:
        metrics["wer"] = float(jiwer_wer(dump["refs"], dump["hyps"]))
        metrics["cer"] = float(jiwer_cer(dump["refs"], dump["hyps"]))
    if dump["y_true"]:
        metrics["f1"] = float(f1_score(dump["y_true"], dump["y_pred"], average="macro", zero_division=0))
        metrics["acc"] = float(accuracy_score(dump["y_true"], dump["y_pred"]))
    return metrics

def pair_clips(dump):
    """One row per clip that has both an ASR and an SER prediction, with its spell power."""
    asr = {p: (r, h) for p, r, h in zip(dump["asr_paths"], dump["refs"], dump["hyps"])}
    rows = []
    for p, t, k, c in zip(dump["ser_paths"], dump["y_true"], dump["y_pred"], dump["y_conf"]):
        if p not in asr: continue
        ref, hyp = asr[p]
        w = clip_wer(ref, hyp)
        rows.append({"path": p, "ref": ref, "hyp": hyp, "true": IDX2EMO[t], "pred": IDX2EMO[k],
                     "conf": c, "wer": w, "power": spell_power(IDX2EMO[k], c, w)})
    return pd.DataFrame(rows)

def _ref_text(batch, i):
    return normalize_text(tokenizer.decode(batch["texts"][i][:batch["text_lens"][i]].tolist(), collapse=False))

@torch.no_grad()
def evaluate_conformer(model, loader, device):
    model.eval()
    dump = _new_dump()
    for batch in tqdm(loader, desc="eval", leave=False):
        ctc_logits, emo_logits, new_lens = model(batch["features"].to(device), batch["feat_lens"].to(device))
        best_path = ctc_logits.argmax(-1).cpu().tolist()
        probs = F.softmax(emo_logits.float(), dim=-1).cpu().numpy()
        for i, task in enumerate(batch["tasks"]):
            if task == "asr":
                ref = _ref_text(batch, i)
                if not ref: continue
                hyp = normalize_text(tokenizer.decode(best_path[i][:int(new_lens[i])], collapse=True))
                dump["refs"].append(ref); dump["hyps"].append(hyp); dump["asr_paths"].append(batch["paths"][i])
            elif task == "ser":
                k = int(probs[i].argmax())
                dump["y_true"].append(int(batch["emotions"][i])); dump["y_pred"].append(k)
                dump["y_conf"].append(float(probs[i, k])); dump["ser_paths"].append(batch["paths"][i])
    return summarize(dump), dump

@torch.no_grad()
def evaluate_whisper(model, loader, device):
    model.eval()
    dump = _new_dump()
    for batch in tqdm(loader, desc="eval", leave=False):
        feats = batch["features"].to(device)
        asr_idx = [i for i, t in enumerate(batch["tasks"]) if t == "asr"]
        ser_idx = [i for i, t in enumerate(batch["tasks"]) if t == "ser"]
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            texts = model.transcribe(feats[asr_idx]) if asr_idx else []          # decode ASR rows only
            emo_logits = (model.emo_head(model.encode(feats[ser_idx]).mean(dim=1))
                          if ser_idx else None)                                 # encode SER rows only
        for k, i in enumerate(asr_idx):
            ref = _ref_text(batch, i)
            if not ref: continue
            dump["refs"].append(ref); dump["hyps"].append(normalize_text(texts[k]))
            dump["asr_paths"].append(batch["paths"][i])
        if ser_idx:
            probs = F.softmax(emo_logits.float(), dim=-1).cpu().numpy()
            for k, i in enumerate(ser_idx):
                j = int(probs[k].argmax())
                dump["y_true"].append(int(batch["emotions"][i])); dump["y_pred"].append(j)
                dump["y_conf"].append(float(probs[k, j])); dump["ser_paths"].append(batch["paths"][i])
    return summarize(dump), dump

# ---------- 8.3 Whisper labels ----------
def build_whisper_labels(batch, device):
    """Token ids for ASR rows (-100 everywhere else). The leading <|startoftranscript|> is dropped
    because Whisper prepends decoder_start_token_id itself when it shifts labels right."""
    tok = whisper_processor.tokenizer
    sot = tok.convert_tokens_to_ids("<|startoftranscript|>")
    seqs = []
    for i, task in enumerate(batch["tasks"]):
        if task != "asr":
            seqs.append(None); continue
        ref_txt = tokenizer.decode(batch["texts"][i][:batch["text_lens"][i]].tolist(), collapse=False)
        ids = tok(ref_txt).input_ids
        if ids and ids[0] == sot:
            ids = ids[1:]
        seqs.append(ids)
    max_len = max([len(s) for s in seqs if s] + [1])
    labels = torch.full((len(seqs), max_len), -100, dtype=torch.long)
    for i, s in enumerate(seqs):
        if s: labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return labels.to(device)

# ---------- 8.4 the trainer ----------
def train_model(model, train_loader, val_loader, *, model_kind: str,
                ckpt_dir: str, epochs: int = 5, lr: float = 3e-4,
                grad_accum: int = 1, resume: bool = True):
    assert model_kind in ("conformer", "whisper")
    model = model.to(DEVICE)
    mtl = MultiTaskUncertaintyLoss(num_tasks=2).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad] + list(mtl.parameters())
    optim = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    steps_per_epoch = max(1, len(train_loader) // grad_accum)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optim, max_lr=lr, epochs=epochs, steps_per_epoch=steps_per_epoch,
        pct_start=0.1, anneal_strategy="cos")
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ckpt = CheckpointManager(ckpt_dir)
    start_epoch, best_score = 0, float("-inf")
    if resume:
        start_epoch, prev = ckpt.resume(model, optim, sched, scaler, mtl, map_location=DEVICE)
        best_score = prev.get("best_score", prev.get("score", float("-inf")))

    ctc_loss = nn.CTCLoss(blank=tokenizer.blank, zero_infinity=True)
    ce_loss = nn.CrossEntropyLoss()
    zero = lambda: torch.zeros((), device=DEVICE)

    for epoch in range(start_epoch, epochs):
        model.train()
        running = {"asr": 0.0, "ser": 0.0, "n_asr": 0, "n_ser": 0}
        pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{epochs}")
        optim.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar):
            feats = batch["features"].to(DEVICE, non_blocking=True)
            emos = batch["emotions"].to(DEVICE)
            tasks = batch["tasks"]
            asr_mask = torch.tensor([t == "asr" for t in tasks], device=DEVICE)
            ser_mask = torch.tensor([t == "ser" for t in tasks], device=DEVICE)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                if model_kind == "conformer":
                    flens = batch["feat_lens"].to(DEVICE)
                    ctc_logits, emo_logits, new_lens = model(feats, flens)
                    L_asr = zero()
                    if asr_mask.any():
                        sel = asr_mask.nonzero(as_tuple=True)[0]
                        tlens = batch["text_lens"].to(DEVICE)[sel]
                        if (tlens > 0).any():
                            log_probs = F.log_softmax(ctc_logits[sel].float(), dim=-1).transpose(0, 1)
                            L_asr = ctc_loss(log_probs, batch["texts"].to(DEVICE)[sel], new_lens[sel], tlens)
                else:  # whisper
                    labels = build_whisper_labels(batch, DEVICE)
                    asr_out, emo_logits = model(feats, labels=labels)
                    L_asr = asr_out.loss if asr_mask.any() else zero()

                L_ser = ce_loss(emo_logits[ser_mask], emos[ser_mask]) if ser_mask.any() else zero()
                loss = mtl([L_asr, L_ser]) / grad_accum

            scaler.scale(loss).backward()
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
                scaler.step(optim); scaler.update()
                optim.zero_grad(set_to_none=True)
                sched.step()

            if asr_mask.any():
                running["asr"] += float(L_asr.detach()) * int(asr_mask.sum())
                running["n_asr"] += int(asr_mask.sum())
            if ser_mask.any():
                running["ser"] += float(L_ser.detach()) * int(ser_mask.sum())
                running["n_ser"] += int(ser_mask.sum())
            w_asr, w_ser = mtl.weights()
            pbar.set_postfix(L_asr=f"{L_asr.item():.3f}", L_ser=f"{L_ser.item():.3f}",
                             w_asr=f"{w_asr:.2f}", w_ser=f"{w_ser:.2f}",
                             lr=f"{sched.get_last_lr()[0]:.1e}")

        avg_asr = running["asr"] / max(1, running["n_asr"])
        avg_ser = running["ser"] / max(1, running["n_ser"])
        print(f"\n— epoch {epoch+1} train: asr={avg_asr:.4f} ser={avg_ser:.4f}")

        eval_fn = evaluate_conformer if model_kind == "conformer" else evaluate_whisper
        metrics, dump = eval_fn(model, val_loader, DEVICE)
        print(f"  VAL  WER={metrics.get('wer', float('nan')):.3f}  "
              f"CER={metrics.get('cer', float('nan')):.3f}  "
              f"F1={metrics.get('f1', float('nan')):.3f}  "
              f"ACC={metrics.get('acc', float('nan')):.3f}")

        # 🔮 Live spell-power demo (ASR + SER of the SAME clip, per-clip WER)
        clips = pair_clips(dump)
        if len(clips):
            r = clips.iloc[random.randrange(len(clips))]
            print(f"  🔮 SPELL DEMO: \"{r['hyp'][:60]}\"  emo={r['pred']} (conf={r['conf']:.2f}, "
                  f"actor meant {r['true']})  WER={r['wer']:.2f}  →  POWER = {r['power']}/100")

        score = metrics.get("f1", 0.0) - metrics.get("wer", 1.0)
        is_best = score > best_score
        if is_best: best_score = score
        ckpt.save(model=model, optimizer=optim, scheduler=sched,
                  scaler=scaler, mtl=mtl, epoch=epoch,
                  metrics={**metrics, "score": score, "best_score": best_score}, is_best=is_best)
    return model

print("✓ trainer ready")

In [ ]:
# ============================================================================
# CELL 8b — Train Model A: Conformer from scratch
# ============================================================================
print("\n" + "=" * 70)
print(" Training Model A — Conformer from scratch")
print("=" * 70)

train_ds_c = SpellEngineDataset(manifest, split="train", augment=True)
val_ds_c = SpellEngineDataset(manifest, split="dev", augment=False)
train_loader_c = DataLoader(train_ds_c, batch_size=16, shuffle=True,
                            num_workers=2, collate_fn=collate_conformer, pin_memory=True)
val_loader_c = DataLoader(val_ds_c, batch_size=16, shuffle=False,
                          num_workers=2, collate_fn=collate_conformer, pin_memory=True)

conformer = ConformerSpellEngine(
    n_mels=N_MELS, d_model=144, num_layers=8, num_heads=4,
    vocab_size=tokenizer.vocab_size, num_emotions=len(EMOTIONS))

EPOCHS_CONF = 5
conformer = train_model(conformer, train_loader_c, val_loader_c,
                        model_kind="conformer", ckpt_dir=DIRS["ckpts_conf"],
                        epochs=EPOCHS_CONF, lr=3e-4)

In [ ]:
# ============================================================================
# CELL 8c — Train Model B: Whisper-Small + LoRA
# ============================================================================
print("\n" + "=" * 70)
print(" Training Model B — Whisper-Small + LoRA")
print("=" * 70)

whisper_model = WhisperSpellEngine(num_emotions=len(EMOTIONS))

train_ds_w = SpellEngineDataset(manifest, split="train", augment=True,
                                use_whisper_features=True,
                                whisper_extractor=whisper_processor.feature_extractor)
val_ds_w = SpellEngineDataset(manifest, split="dev", augment=False,
                              use_whisper_features=True,
                              whisper_extractor=whisper_processor.feature_extractor)
train_loader_w = DataLoader(train_ds_w, batch_size=4, shuffle=True,
                            num_workers=2, collate_fn=collate_whisper, pin_memory=True)
val_loader_w = DataLoader(val_ds_w, batch_size=8, shuffle=False,
                          num_workers=2, collate_fn=collate_whisper, pin_memory=True)

EPOCHS_WHIS = 3
whisper_model = train_model(whisper_model, train_loader_w, val_loader_w,
                            model_kind="whisper", ckpt_dir=DIRS["ckpts_whis"],
                            epochs=EPOCHS_WHIS, lr=1e-4, grad_accum=4)

## CELL 9 — Evaluation & Comparison Report

Loads each model's `best.pt`, runs the **test** split, builds a Pandas comparison table and side-by-side confusion matrices.

> **Reading the numbers:** CREMA-D uses only 12 sentences, and with `SPLIT_BY = "file"` the same 91 actors appear in every split.
> WER here therefore measures *recognising known incantations from known voices* — exactly the game's use case, but optimistic as a
> general ASR figure. Use `SPLIT_BY = "speaker"` for an honest unseen-player estimate of emotion accuracy.

In [ ]:
# ============================================================================
# CELL 9 — Evaluation & Comparison Report
# ============================================================================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns, matplotlib.pyplot as plt

def count_params_m(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only) / 1e6

def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

test_ds_c = SpellEngineDataset(manifest, split="test", augment=False)
test_loader_c = DataLoader(test_ds_c, batch_size=16, shuffle=False,
                           num_workers=2, collate_fn=collate_conformer)
test_ds_w = SpellEngineDataset(manifest, split="test", augment=False,
                               use_whisper_features=True,
                               whisper_extractor=whisper_processor.feature_extractor)
test_loader_w = DataLoader(test_ds_w, batch_size=8, shuffle=False,
                           num_workers=2, collate_fn=collate_whisper)

load_weights(conformer, Path(DIRS["ckpts_conf"]) / "best.pt")
load_weights(whisper_model, Path(DIRS["ckpts_whis"]) / "best.pt")

print("Evaluating Conformer …")
conf_metrics, conf_dump = evaluate_conformer(conformer.to(DEVICE), test_loader_c, DEVICE)
print("Evaluating Whisper-LoRA …")
whis_metrics, whis_dump = evaluate_whisper(whisper_model.to(DEVICE), test_loader_w, DEVICE)

report = pd.DataFrame({
    "Model":                ["Conformer (scratch)", "Whisper-Small + LoRA"],
    "Total params (M)":     [count_params_m(conformer), count_params_m(whisper_model)],
    "Trainable params (M)": [count_params_m(conformer, True), count_params_m(whisper_model, True)],
    "Size (MB)":            [model_size_mb(conformer), model_size_mb(whisper_model)],
    "WER ↓":     [conf_metrics.get("wer", float("nan")), whis_metrics.get("wer", float("nan"))],
    "CER ↓":     [conf_metrics.get("cer", float("nan")), whis_metrics.get("cer", float("nan"))],
    "Emo Acc ↑": [conf_metrics.get("acc", float("nan")), whis_metrics.get("acc", float("nan"))],
    "Emo F1 ↑":  [conf_metrics.get("f1",  float("nan")), whis_metrics.get("f1",  float("nan"))],
}).round(4)

print("\n" + "=" * 70)
print("FINAL COMPARISON — SpellEngine ASR + SER (test split)")
print("=" * 70)
print(report.to_string(index=False))
report.to_csv(Path(DIRS["logs"]) / "final_report.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, dump) in zip(axes, [("Conformer", conf_dump), ("Whisper-LoRA", whis_dump)]):
    if not dump["y_true"]:
        ax.set_visible(False); continue
    cm = confusion_matrix(dump["y_true"], dump["y_pred"], labels=list(range(len(EMOTIONS))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax)
    ax.set_title(f"{name} — emotion confusion")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
plt.savefig(Path(DIRS["logs"]) / "confusion_matrices.png", dpi=120)
plt.show()

for name, dump in [("Conformer", conf_dump), ("Whisper-LoRA", whis_dump)]:
    if dump["y_true"]:
        print(f"\n— {name} per-class —")
        print(classification_report(dump["y_true"], dump["y_pred"], labels=list(range(len(EMOTIONS))),
                                    target_names=EMOTIONS, zero_division=0))

## CELL 10 — Export for the Game Runtime

- **Conformer → ONNX** (`exports/conformer_spellengine.onnx`), with dynamic batch and length axes. The graph returns game-ready
  outputs: CTC log-probabilities, emotion probabilities and valid output lengths. A parity check compares ONNX Runtime with PyTorch
  on a padded batch of real test clips and reports CPU latency.
- **Whisper → LoRA adapter** (`exports/whisper_lora_adapter/`, a few MB) + the SER head + processor, re-attachable to the public
  `openai/whisper-small` weights.
- **`spellengine_meta.json`** — feature settings, vocabulary, emotion order and intent weights, so a game engine can reproduce
  the preprocessing exactly.

In [ ]:
# ============================================================================
# CELL 10 — Export: Conformer → ONNX (+ parity check), Whisper → LoRA adapter
# ============================================================================
import copy, json, time
import onnx, onnxruntime as ort

EXPORT_DIR = Path(DIRS["exports"])

class ConformerForExport(nn.Module):
    """Game-ready graph: CTC log-probs, emotion probabilities, valid output lengths."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, features, feat_lens):
        ctc_logits, emo_logits, out_lens = self.model(features, feat_lens)
        return F.log_softmax(ctc_logits, dim=-1), F.softmax(emo_logits, dim=-1), out_lens

export_model = ConformerForExport(copy.deepcopy(conformer).float().cpu()).eval()
onnx_path = EXPORT_DIR / "conformer_spellengine.onnx"

ex = test_ds_c[0]
dummy = (ex["features"].unsqueeze(0), torch.tensor([ex["feat_len"]], dtype=torch.long))
export_kwargs = dict(
    input_names=["features", "feat_lens"],
    output_names=["ctc_log_probs", "emo_probs", "out_lens"],
    dynamic_axes={"features": {0: "batch", 2: "frames"}, "feat_lens": {0: "batch"},
                  "ctc_log_probs": {0: "batch", 1: "steps"}, "emo_probs": {0: "batch"},
                  "out_lens": {0: "batch"}},
    opset_version=17,
)

def export_conformer_onnx():
    # nn.MultiheadAttention's fused inference "fast path" has no ONNX mapping — trace the plain ops.
    mha = getattr(torch.backends, "mha", None)
    prev = mha.get_fastpath_enabled() if mha else None
    if mha: mha.set_fastpath_enabled(False)
    try:
        with torch.no_grad():
            # TorchScript exporter first, then the torch.export-based one, then the torch default
            for name, extra in (("TorchScript", {"dynamo": False}), ("dynamo", {"dynamo": True}), ("default", {})):
                try:
                    torch.onnx.export(export_model, dummy, str(onnx_path), **extra, **export_kwargs)
                    return name
                except Exception as e:
                    print(f"  {name} exporter failed: {type(e).__name__}: {str(e)[:300]}")
    finally:
        if mha: mha.set_fastpath_enabled(prev)
    return None

exporter = export_conformer_onnx()
if exporter is None:
    print("✗ ONNX export failed — see messages above (the PyTorch checkpoint is unaffected).")
else:
    onnx.checker.check_model(str(onnx_path))
    print(f"✓ {exporter} exporter → {onnx_path.name} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    # Parity on a padded batch of two clips whose lengths differ from the export dummy
    pb = collate_conformer([test_ds_c[i] for i in (1, 2)])
    with torch.no_grad():
        pt_out = [t.numpy() for t in export_model(pb["features"], pb["feat_lens"])]
    ort_out = sess.run(None, {"features": pb["features"].numpy(), "feat_lens": pb["feat_lens"].numpy()})

    lens = pt_out[2]
    ctc_diff = max(np.abs(pt_out[0][i, :lens[i]] - ort_out[0][i, :lens[i]]).max() for i in range(len(lens)))
    emo_diff = np.abs(pt_out[1] - ort_out[1]).max()
    greedy_same = all((pt_out[0][i, :lens[i]].argmax(-1) == ort_out[0][i, :lens[i]].argmax(-1)).all()
                      for i in range(len(lens)))
    print(f"  parity: max|Δ ctc log-prob|={ctc_diff:.2e}  max|Δ emo prob|={emo_diff:.2e}  "
          f"out_lens equal={np.array_equal(lens, ort_out[2])}  greedy CTC identical={greedy_same}")

    # CPU latency for one clip
    one = {"features": pb["features"][:1].numpy(), "feat_lens": pb["feat_lens"][:1].numpy()}
    for _ in range(3): sess.run(None, one)
    t0 = time.perf_counter()
    for _ in range(20): sess.run(None, one)
    ort_ms = (time.perf_counter() - t0) / 20 * 1e3
    with torch.no_grad():
        for _ in range(3): export_model(pb["features"][:1], pb["feat_lens"][:1])
        t0 = time.perf_counter()
        for _ in range(20): export_model(pb["features"][:1], pb["feat_lens"][:1])
    pt_ms = (time.perf_counter() - t0) / 20 * 1e3
    secs = int(pb["feat_lens"][0]) * HOP / SR
    print(f"  CPU latency for a {secs:.1f}s clip: ONNX Runtime {ort_ms:.1f} ms | PyTorch {pt_ms:.1f} ms")

# ---- Whisper: save only the LoRA adapter + SER head + processor ----
adapter_dir = EXPORT_DIR / "whisper_lora_adapter"
whisper_model.whisper.save_pretrained(str(adapter_dir))
torch.save(whisper_model.emo_head.state_dict(), adapter_dir / "emo_head.pt")
whisper_processor.save_pretrained(str(adapter_dir))
adapter_mb = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / 1e6
print(f"✓ Whisper LoRA adapter + SER head → {adapter_dir.name}/ ({adapter_mb:.1f} MB)")
# Re-attach later with:
#   base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
#   peft_model = PeftModel.from_pretrained(base, adapter_dir)      # from peft import PeftModel

# ---- Metadata the game runtime needs to reproduce preprocessing ----
meta = {
    "sample_rate": SR,
    "conformer_features": {
        "transform": "torchaudio MelSpectrogram(power=2, center=True) → AmplitudeToDB(top_db=80) → per-utterance z-norm",
        "n_fft": N_FFT, "hop_length": HOP, "n_mels": N_MELS},
    "ctc_vocab": tokenizer.itos, "ctc_blank": tokenizer.blank,
    "emotions": EMOTIONS,
    "intent_weights": INTENT_WEIGHTS,
    "spell_power": "100 * intent_weight * emotion_conf * (0.4 + 0.6 * (1 - min(WER, 1)))",
}
(EXPORT_DIR / "spellengine_meta.json").write_text(json.dumps(meta, indent=2))
print(f"✓ metadata → spellengine_meta.json")

## CELL 11 — 🔮 Cast a Spell

`cast_spell(wav_path, incantation, engine)` is the game loop in one function: transcribe, read the emotion, score clarity against
the incantation the player was supposed to say, and return the Spell Power. Below it is run on one test clip per emotion with both
engines, then aggregated over the whole test split.

**Try your own voice:** record a WAV, add it as a Kaggle dataset (or upload it to `/kaggle/working/`), then run
`cast_spell("/kaggle/input/<your-dataset>/me.wav", "it's eleven o'clock")`.

In [ ]:
# ============================================================================
# CELL 11 — Cast a Spell (inference demo)
# ============================================================================
@torch.no_grad()
def cast_spell(wav_path, incantation, engine="conformer"):
    wav = load_audio(wav_path)
    if engine == "conformer":
        conformer.eval()
        feats = conformer_features(wav).unsqueeze(0).to(DEVICE)
        ctc_logits, emo_logits, lens = conformer(feats, torch.tensor([feats.shape[-1]], device=DEVICE))
        heard = tokenizer.decode(ctc_logits[0, :int(lens[0])].argmax(-1).tolist(), collapse=True)
    elif engine == "whisper":
        whisper_model.eval()
        feats = whisper_processor.feature_extractor(
            wav.numpy(), sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE)
        heard = whisper_model.transcribe(feats)[0]
        emo_logits = whisper_model.emo_head(whisper_model.encode(feats).mean(dim=1))
    else:
        raise ValueError("engine must be 'conformer' or 'whisper'")

    probs = F.softmax(emo_logits.float(), dim=-1)[0].cpu()
    k = int(probs.argmax())
    emotion, conf = IDX2EMO[k], float(probs[k])
    heard = normalize_text(heard)
    w = clip_wer(normalize_text(incantation), heard)
    return {"engine": engine, "heard": heard, "emotion": emotion,
            "confidence": round(conf, 3), "wer": round(w, 3),
            "power": spell_power(emotion, conf, w)}

# ---- One test clip per emotion, both engines ----
demo_clips = (manifest[(manifest["split"] == "test") & (manifest["task"] == "ser")]
              .groupby("emotion", group_keys=False).sample(1, random_state=SEED))
results = []
for _, row in demo_clips.iterrows():
    for engine in ("conformer", "whisper"):
        r = cast_spell(row["path"], row["text"], engine)
        results.append({"actor meant": row["emotion"], "incantation": row["text"], **r})
demo_df = pd.DataFrame(results)
print(demo_df.to_string(index=False))

# ---- Spell power over the whole test split (reuses Cell 9's predictions) ----
frames = []
for name, dump in [("Conformer", conf_dump), ("Whisper-LoRA", whis_dump)]:
    df = pair_clips(dump)
    df["engine"] = name
    frames.append(df)
clips = pd.concat(frames, ignore_index=True)

order = sorted(EMOTIONS, key=lambda e: -INTENT_WEIGHTS[e])
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=clips, x="true", y="power", hue="engine", order=order, errorbar=None, ax=ax)
ax.set_title("Mean spell power by the emotion the actor intended (test split)")
ax.set_xlabel("intended emotion (sorted by intent weight)"); ax.set_ylabel("spell power (0–100)")
plt.tight_layout()
plt.savefig(Path(DIRS["logs"]) / "spell_power_by_emotion.png", dpi=120)
plt.show()

summary = clips.groupby(["engine", "true"])["power"].mean().unstack().reindex(columns=order).round(1)
print(summary.to_string())
clips.to_csv(Path(DIRS["logs"]) / "test_spell_power.csv", index=False)